# PTM Binder Design with Artisan

This notebook mirrors the phosphopeptide binder-design tutorial style, but it keeps the actual model work inside `ptm_foundry` and uses `artisan` only for pipeline orchestration.

We will run the same workshop target used in the Colab notebook:

- target peptide: `PVPNPD(PTR)EPIRKGQ`
- target chain: `B`
- binder chain: `A`
- stages: spoof target -> RFD3 -> LigandMPNN -> RF3 -> tutorial-style metrics -> filter
        


## 0. One-Time Local Setup

Use this notebook with a local Jupyter kernel, not Colab. The notebook assumes:

- `ptm_foundry` and `artisan` live side-by-side
- the `artisan` repo is checked out at `../artisan`
- a Prefect server is running before `PipelineManager.create(...)`

Recommended shell setup from the `ptm_foundry` repo root:

```bash
git clone --branch release/v0.1.2a2 https://github.com/dexterity-systems/artisan ../artisan
pixi install
pixi run python -m pip install prefect[dask] prefect-submitit submitit asyncpg deltalake polars xxhash ipywidgets rootutils rich beartype hydride
pixi run python -m ipykernel install --user --name ptm-artisan --display-name "PTM Foundry + Artisan"
```

Start a Prefect server in another terminal before creating the pipeline. One of these should work depending on your environment:

```bash
pixi run prefect-server start --bg
# or
pixi run prefect server start
```
        


In [ ]:
from pathlib import Path
import shutil
import sys
import warnings

warnings.filterwarnings("ignore", module="atomworks")

cwd = Path.cwd().resolve()
REPO_DIR = next(
    candidate
    for candidate in [cwd, *cwd.parents]
    if (candidate / "examples").exists() and (candidate / "models").exists()
)
ARTISAN_REPO = REPO_DIR.parent / "artisan"
EXAMPLES_DIR = REPO_DIR / "examples"

SRC_PATHS = [
    REPO_DIR / "src",
    REPO_DIR / "models" / "rfd3" / "src",
    REPO_DIR / "models" / "mpnn" / "src",
    REPO_DIR / "models" / "rf3" / "src",
    EXAMPLES_DIR,
    ARTISAN_REPO / "src",
]
for path in SRC_PATHS:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"Repository root: {REPO_DIR}")
print(f"Artisan repo:    {ARTISAN_REPO}")
print(f"prefect:         {shutil.which('prefect')}")
print(f"prefect-server:  {shutil.which('prefect-server')}")
print(f"PREFECT_API_URL: {__import__('os').environ.get('PREFECT_API_URL', '<not set>')}")
        


In [ ]:
import subprocess

BOOTSTRAP_ARTISAN = False
ARTISAN_PIP_DEPS = [
    "prefect[dask]>=3.6,<4.0",
    "prefect-submitit>=0.1.4",
    "submitit",
    "asyncpg",
    "deltalake>=0.14.0",
    "polars>=0.20.0",
    "xxhash",
    "ipywidgets",
    "rootutils>=1.0",
    "rich",
    "beartype",
    "hydride",
]

if BOOTSTRAP_ARTISAN:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *ARTISAN_PIP_DEPS])
else:
    print("Set BOOTSTRAP_ARTISAN = True if this kernel still needs the Artisan runtime dependencies.")
        


In [ ]:
import json

import numpy as np
import pandas as pd
from IPython.display import display
from lightning.fabric import seed_everything

from artisan.operations.curator import Filter
from artisan.orchestration import Backend, PipelineManager
from artisan.schemas.artifact.registry import ArtifactTypeDef
from artisan.visualization.inspect import inspect_metrics, inspect_pipeline
from atomworks.io.utils.visualize import view
from foundry.inference_engines.checkpoint_registry import REGISTERED_CHECKPOINTS
from spoof_cif import (
    bond_table_for_residue,
    chain_summary,
    plot_local_atom_bond_graph,
    plot_residue_bond_graph,
)
from ptm_artisan_ops import (
    BinderAlignedRMSD,
    PhosphositeHBondMetrics,
    RunLigandMPNNDesign,
    RunRFD3Design,
    RunRF3Refold,
    SelectionSASAMetrics,
    SpoofPTMTarget,
    load_atom_array,
    resolve_file_ref_paths,
)

for key in ["rfd3", "ligandmpnn", "rf3"]:
    ckpt_path = REGISTERED_CHECKPOINTS[key].get_default_path()
    print(f"{key:12s} -> {ckpt_path} (exists={Path(ckpt_path).exists()})")
        


## 1. Configure the Workshop Run

These parameters mirror the Colab workshop notebook, but we now materialize all pipeline outputs under `examples/runs/ptm_artisan_pipeline/` so the file-reference artifacts remain usable after each step finishes.
        


In [ ]:
seed_everything(7)

TARGET_SEQUENCE = "PVPNPD(PTR)EPIRKGQ"
TARGET_CHAIN_ID = "B"
BINDER_CHAIN_ID = "A"
BINDER_LENGTH = 100
EXAMPLE_NAME = "pvpnpd_ptr_workshop"
PIPELINE_NAME = "ptm_artisan_pipeline"

RUNS_DIR = REPO_DIR / "examples" / "runs" / PIPELINE_NAME
DELTA_ROOT = RUNS_DIR / "delta"
STAGING_ROOT = RUNS_DIR / "staging"
WORKING_ROOT = RUNS_DIR / "working"
MATERIALIZED_ROOT = RUNS_DIR / "materialized"

SPOOF_OUTPUT_DIR = MATERIALIZED_ROOT / "spoof_target"
RFD3_OUTPUT_DIR = MATERIALIZED_ROOT / "rfd3"
MPNN_OUTPUT_DIR = MATERIALIZED_ROOT / "mpnn"
RF3_OUTPUT_DIR = MATERIALIZED_ROOT / "rf3"

for path in [
    DELTA_ROOT,
    STAGING_ROOT,
    WORKING_ROOT,
    SPOOF_OUTPUT_DIR,
    RFD3_OUTPUT_DIR,
    MPNN_OUTPUT_DIR,
    RF3_OUTPUT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

RFD3_PARAMS = {
    "ckpt_path": "rfd3",
    "diffusion_batch_size": 1,
    "n_batches": 1,
    "output_dir": str(RFD3_OUTPUT_DIR),
}
MPNN_PARAMS = {
    "checkpoint_path": "ligandmpnn",
    "batch_size": 4,
    "designed_chains": [BINDER_CHAIN_ID],
    "target_chain_id": TARGET_CHAIN_ID,
    "remove_waters": True,
    "is_legacy_weights": True,
    "output_dir": str(MPNN_OUTPUT_DIR),
}
RF3_PARAMS = {
    "ckpt_path": "rf3",
    "annotate_b_factor_with_plddt": True,
    "output_dir": str(RF3_OUTPUT_DIR),
}
FILTER_CRITERIA = [
    {"metric": "whole_peptide_ca_rmsd", "operator": "lt", "value": 2.0},
    {"metric": "po4.fraction_buried", "operator": "gt", "value": 0.85},
    {"metric": "phosphosite_hbonds", "operator": "ge", "value": 0.0},
]

print(f"Target sequence: {TARGET_SEQUENCE}")
print(f"Run directory:   {RUNS_DIR}")
print(json.dumps({
    "rfd3": RFD3_PARAMS,
    "mpnn": MPNN_PARAMS,
    "rf3": RF3_PARAMS,
    "filter": FILTER_CRITERIA,
}, indent=2))
        


## 2. Create the Pipeline

This is the same top-level orchestration pattern used in the Artisan docs: create a `PipelineManager`, capture the `output` helper, then run one step per cell.
        


In [ ]:
pipeline = PipelineManager.create(
    name=PIPELINE_NAME,
    delta_root=DELTA_ROOT,
    staging_root=STAGING_ROOT,
    working_root=WORKING_ROOT,
    preserve_working=False,
    backend=Backend.LOCAL,
)
output = pipeline.output
pipeline
        


## 3. Spoof the PTM Target

This step uses the same `build_ptm_binder_workshop_inputs()` logic from `ptm_foundry/examples/spoof_cif.py`. It creates:

- a spoofed CIF for `PVPNPD(PTR)EPIRKGQ`
- the matching RFD3 JSON config
- compact metadata with the PTR residue index
        


In [ ]:
spoof_step = pipeline.run(
    operation=SpoofPTMTarget,
    name="spoof_target",
    params={
        "sequence": TARGET_SEQUENCE,
        "binder_length": BINDER_LENGTH,
        "target_chain_id": TARGET_CHAIN_ID,
        "ptm_resname": "PTR",
        "example_name": EXAMPLE_NAME,
        "output_dir": str(SPOOF_OUTPUT_DIR),
    },
    backend=Backend.LOCAL,
)
spoof_step
        


In [ ]:
spoofed_target_path = Path(spoof_step.metadata["cif_path"])
ptm_residue_id = int(spoof_step.metadata["ptm_residue_id"])
spoofed_target = load_atom_array(spoofed_target_path, hydrogen_policy="remove")

display(chain_summary(spoofed_target))
print(f"Spoofed target path: {spoofed_target_path}")
print(f"PTR residue id:      {ptm_residue_id}")
view(spoofed_target)
        


In [ ]:
bond_table = bond_table_for_residue(
    spoofed_target,
    chain_id=TARGET_CHAIN_ID,
    residue_id=ptm_residue_id,
    include_neighbors=True,
)
display(bond_table)
        


In [ ]:
plot_residue_bond_graph(
    spoofed_target,
    chain_id=TARGET_CHAIN_ID,
    residue_id=ptm_residue_id,
)
        


In [ ]:
plot_local_atom_bond_graph(
    spoofed_target,
    chain_id=TARGET_CHAIN_ID,
    residue_id=ptm_residue_id,
)
        


## 4. Generate Binder Backbones with RFD3

The `rfd3` step consumes the spoofed CIF plus the RFD3 JSON config and materializes one or more designed complexes.
        


In [ ]:
rfd3_step = pipeline.run(
    operation=RunRFD3Design,
    name="rfd3",
    inputs={
        "structures": output("spoof_target", "structures"),
        "config": output("spoof_target", "config"),
    },
    params=RFD3_PARAMS,
    backend=Backend.LOCAL,
)
rfd3_step
        


In [ ]:
rfd3_paths = [Path(path) for path in rfd3_step.metadata["structure_paths"]]
rfd3_complex = load_atom_array(rfd3_paths[0], hydrogen_policy="remove")

print("RFD3 outputs:")
for path in rfd3_paths:
    print(" -", path)

display(chain_summary(rfd3_complex))
display(inspect_metrics(DELTA_ROOT, rfd3_step.step_number))
view(rfd3_complex)
        


## 5. Design the Binder Sequence with LigandMPNN

This step keeps the peptide chain fixed and redesigns only binder chain `A`.
        


In [ ]:
mpnn_step = pipeline.run(
    operation=RunLigandMPNNDesign,
    name="mpnn",
    inputs={"structures": output("rfd3", "structures")},
    params=MPNN_PARAMS,
    backend=Backend.LOCAL,
)
mpnn_step
        


In [ ]:
mpnn_metrics = inspect_metrics(DELTA_ROOT, mpnn_step.step_number)
display(mpnn_metrics)

mpnn_paths = [Path(path) for path in mpnn_step.metadata["structure_paths"]]
selected_mpnn_path = mpnn_paths[0]
selected_complex = load_atom_array(selected_mpnn_path, hydrogen_policy="remove")

print(f"Selected MPNN design: {selected_mpnn_path}")
view(selected_complex)
        


## 6. Refold the Designs with RF3

The RF3 step refolds each LigandMPNN-designed complex and records the standard RF3 summary confidence metrics.
        


In [ ]:
rf3_step = pipeline.run(
    operation=RunRF3Refold,
    name="rf3",
    inputs={"structures": output("mpnn", "structures")},
    params=RF3_PARAMS,
    backend=Backend.LOCAL,
)
rf3_step
        


In [ ]:
rf3_metrics = inspect_metrics(DELTA_ROOT, rf3_step.step_number)
display(rf3_metrics)

rf3_paths = [Path(path) for path in rf3_step.metadata["structure_paths"]]
first_rf3_path = rf3_paths[0]
first_rf3_complex = load_atom_array(first_rf3_path, hydrogen_policy="remove")

print(f"First RF3 output: {first_rf3_path}")
view(first_rf3_complex)
        


## 7. Tutorial-Style Post-RF3 Metrics

These three steps reproduce the key tutorial metrics after RF3:

- binder-backbone-aligned peptide and phosphosite RMSDs
- phosphosite hydrogen bonds
- phosphate / PTR burial by SASA
        


In [ ]:
rmsd_step = pipeline.run(
    operation=BinderAlignedRMSD,
    name="rmsd_metrics",
    inputs={
        "reference": output("mpnn", "structures"),
        "mobile": output("rf3", "structures"),
    },
    params={
        "target_chain_id": TARGET_CHAIN_ID,
        "ptm_residue_id": ptm_residue_id,
        "ptm_resname": "PTR",
        "binder_chain_id": BINDER_CHAIN_ID,
    },
    backend=Backend.LOCAL,
)
rmsd_step
        


In [ ]:
rmsd_metrics = inspect_metrics(DELTA_ROOT, rmsd_step.step_number)
display(rmsd_metrics)
        


In [ ]:
hbond_step = pipeline.run(
    operation=PhosphositeHBondMetrics,
    name="hbond_metrics",
    inputs={"structures": output("rf3", "structures")},
    params={
        "target_chain_id": TARGET_CHAIN_ID,
        "ptm_residue_id": ptm_residue_id,
        "ptm_resname": "PTR",
    },
    backend=Backend.LOCAL,
)
hbond_step
        


In [ ]:
hbond_metrics = inspect_metrics(DELTA_ROOT, hbond_step.step_number)
display(hbond_metrics)
        


In [ ]:
sasa_step = pipeline.run(
    operation=SelectionSASAMetrics,
    name="sasa_metrics",
    inputs={"structures": output("rf3", "structures")},
    params={
        "target_chain_id": TARGET_CHAIN_ID,
        "ptm_residue_id": ptm_residue_id,
        "ptm_resname": "PTR",
    },
    backend=Backend.LOCAL,
)
sasa_step
        


In [ ]:
sasa_metrics = inspect_metrics(DELTA_ROOT, sasa_step.step_number)
display(sasa_metrics)
        


In [ ]:
final_metric_table = (
    inspect_metrics(DELTA_ROOT, rmsd_step.step_number)
    .join(
        inspect_metrics(DELTA_ROOT, hbond_step.step_number).select("name", "phosphosite_hbonds"),
        on="name",
        how="left",
    )
    .join(
        inspect_metrics(DELTA_ROOT, sasa_step.step_number).select(
            "name",
            "po4.fraction_buried",
            "ptr.fraction_buried",
        ),
        on="name",
        how="left",
    )
)
display(final_metric_table)
        


## 8. Filter the RF3 Outputs

This uses Artisan's built-in `Filter` operation to keep only RF3 structures that satisfy the workshop thresholds.
        


In [ ]:
filter_step = pipeline.run(
    operation=Filter,
    name="filter",
    inputs={"passthrough": output("rf3", "structures")},
    params={"criteria": FILTER_CRITERIA},
    backend=Backend.LOCAL,
)
filter_step
        


In [ ]:
finalize_summary = pipeline.finalize()
print(json.dumps(finalize_summary, indent=2))
        


## 9. Inspect the Pipeline and Final Structures

`inspect_pipeline()` gives the step-level overview. Then we resolve the final `Filter` output reference back to concrete file paths so you can open the surviving CIFs.
        


In [ ]:
inspect_pipeline(DELTA_ROOT)
        


In [ ]:
filtered_paths = resolve_file_ref_paths(DELTA_ROOT, filter_step.output("passthrough"))
filtered_df = pd.DataFrame({"filtered_cif": [str(path) for path in filtered_paths]})
display(filtered_df)
print("Filter diagnostics:")
print(json.dumps(filter_step.metadata, indent=2, default=float))
        


In [ ]:
if filtered_paths:
    final_path = filtered_paths[0]
    final_atom_array = load_atom_array(final_path, hydrogen_policy="remove")
    print(f"Viewing top surviving RF3 complex: {final_path}")
    view(final_atom_array)
else:
    print("No RF3 structures passed the final filter in this run.")
        
